# 04 · Reconhecimento de entidades nomeadas (NER)

Entrega da Etapa 3 (29/10). O notebook:

1. define as entidades que importam para o radar: empresas e outras organizações (`ORG`), códigos da B3 (`TICKER`), pessoas, locais, valores, percentuais e períodos;
2. compara quatro reconhecedores, cada um somando uma camada ao anterior:
   - **A** · spaCy `pt_core_news_sm` puro, como na aula 9;
   - **B** · spaCy `pt_core_news_lg` puro (modelo maior, com vetores de palavras);
   - **C** · B + `EntityRuler` com os nomes da carteira e o padrão de ticker + expressões regulares para valores, percentuais e períodos;
   - **D** · C + lista de instituições do mercado (bancos, casas de análise, órgãos públicos) + correções pelo contexto: nome seguido de "(TICKER)" é empresa, e siglas como JCP e ROE não são entidade;
3. avalia os quatro com precisão, revocação e F1 **por entidade**, contra 60 manchetes anotadas à mão (`data/ner_ouro.csv`, guia em `Entregas/30-10-2026/guia_ner.md`);
4. mede se o NER encontra **a empresa certa da carteira** nas 200 manchetes de teste (ligação de entidades);
5. lista os erros e junta as entidades com os rótulos de sentimento, que é o que o chatbot vai consultar.

O código dos reconhecedores fica em `src/ner.py`, para ser reaproveitado na aplicação.

**Como rodar:** abra no VS Code ou no Jupyter e execute tudo, a partir da pasta `notebooks/`. Roda em CPU em menos de um minuto. Antes, instale os modelos do spaCy:
`python -m spacy download pt_core_news_sm` e `python -m spacy download pt_core_news_lg`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from spacy import displacy

sys.path.insert(0, "../src")
from ner import carrega_nlp, empresas_citadas, entidades, normaliza

DADOS = Path("../data")
SAIDA = Path("../Entregas/30-10-2026")
SAIDA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

## Dados

As 200 manchetes revisadas e, para 60 delas, as entidades anotadas. A anotação guarda só texto e rótulo; a posição de cada entidade na manchete é recuperada aqui.

In [2]:
manchetes = pd.read_csv(DADOS / "teste_revisao.csv")
ouro_csv = pd.read_csv(DADOS / "ner_ouro.csv").merge(manchetes[["id", "manchete"]], on="id")


def posiciona(manchete, anotacao):
    """'Itaú|ORG;ITUB4|TICKER' -> lista de dicts com texto, rótulo e posição na manchete."""
    texto, ents, inicio_busca = normaliza(manchete), [], {}
    for item in anotacao.split(";"):
        trecho, rotulo = item.rsplit("|", 1)
        trecho = normaliza(trecho)
        ini = texto.find(trecho, inicio_busca.get(trecho, 0))
        assert ini >= 0, f"'{trecho}' não está em: {manchete}"
        inicio_busca[trecho] = ini + len(trecho)
        ents.append({"texto": trecho, "rotulo": rotulo, "inicio": ini, "fim": ini + len(trecho)})
    return ents


ouro = {l["id"]: posiciona(l["manchete"], l["entidades"]) for _, l in ouro_csv.iterrows()}
ROTULOS = ["ORG", "TICKER", "PESSOA", "LOCAL", "VALOR", "PERCENTUAL", "PERIODO"]

contagem = pd.Series([e["rotulo"] for ents in ouro.values() for e in ents]).value_counts().reindex(ROTULOS)
print(f"{len(ouro)} manchetes anotadas, {contagem.sum()} entidades")
contagem.to_frame("entidades").T

60 manchetes anotadas, 205 entidades


,ORG,TICKER,PESSOA,LOCAL,VALOR,PERCENTUAL,PERIODO
entidades,99,37,8,9,18,16,18


## Os quatro reconhecedores

No C, o `EntityRuler` entra **antes** do `ner` estatístico: os nomes da carteira e os tickers ficam travados e o modelo só completa o resto. Cada padrão de empresa leva o ticker como `id`, então "Axia Energia", "Eletrobras" e "ELET6" apontam todos para ELET3. Os apelidos ambíguos da carteira ("Vale", "Vivo", "Localiza") só casam com a grafia exata, como no filtro do notebook 01. O tokenizador também passa a separar o prefixo "ex-", para "ex-Eletrobras" casar com "Eletrobras".

Valores, percentuais e períodos vêm de expressões regulares, como no exercício de valores em R$ da aula 9. O spaCy em português foi treinado no WikiNER, que só tem `PER`, `ORG`, `LOC` e `MISC`, então nem A nem B reconhecem esses tipos.

O D nasceu da análise de erros do C: o modelo estatístico marcava empresas fora da carteira como `LOCAL` ("Cemig", "Porto Seguro", "Mercado Livre") e siglas como `ORG` ("JCP", "ROE"). **Cuidado ao ler o resultado do D:** a lista de instituições foi montada com conhecimento do mercado, mas parte dos nomes também aparece nas 60 manchetes anotadas, então o ganho do D nessa amostra tende a ser otimista. A regra do "(TICKER)" e a lista de siglas não dependem da amostra.

In [3]:
CARTEIRA = DADOS / "carteira.csv"
SISTEMAS = {
    "A · sm puro": carrega_nlp(CARTEIRA, modelo="pt_core_news_sm", com_regras=False),
    "B · lg puro": carrega_nlp(CARTEIRA, modelo="pt_core_news_lg", com_regras=False),
    "C · B + carteira + regras": carrega_nlp(CARTEIRA, com_mercado=False),
    "D · C + mercado": carrega_nlp(CARTEIRA),
}
nlp = SISTEMAS["D · C + mercado"]
print(nlp.pipe_names)

exemplo = manchetes.set_index("id").loc["M0039", "manchete"]
for nome, n in SISTEMAS.items():
    print(f"{nome:<32}", [(e["texto"], e["rotulo"]) for e in entidades(n, exemplo)])

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'entity_ruler', 'ner', 'regras_financeiras', 'regras_mercado']
A · sm puro                      [('Itaú', 'LOCAL'), ('ITUB4', 'LOCAL')]
B · lg puro                      [('Itaú', 'ORG'), ('ITUB4', 'ORG')]
C · B + carteira + regras        [('Itaú', 'ORG'), ('ITUB4', 'TICKER'), ('R$ 12,4 bilhões', 'VALOR'), ('2T26', 'PERIODO'), ('7,8%', 'PERCENTUAL')]
D · C + mercado                  [('Itaú', 'ORG'), ('ITUB4', 'TICKER'), ('R$ 12,4 bilhões', 'VALOR'), ('2T26', 'PERIODO'), ('7,8%', 'PERCENTUAL')]


In [4]:
CORES = {"ORG": "#7aecec", "TICKER": "#bfeeb7", "PESSOA": "#aa9cfc", "LOCAL": "#ff9561",
         "VALOR": "#e4e7d2", "PERCENTUAL": "#ffeb80", "PERIODO": "#c887fb"}
displacy.render(nlp(normaliza(exemplo)), style="ent", jupyter=True, options={"colors": CORES})

## Avaliação por entidade

- **Exata:** acerto só se texto e rótulo forem iguais aos da anotação, ignorando maiúsculas.
- **Relaxada:** acerto se o rótulo é o mesmo e o trecho se sobrepõe ao anotado. Separa erros de fronteira de erros de verdade.

Cada entidade anotada pode ser usada para validar uma única previsão.

In [5]:
def compara(previstas, anotadas, relaxada=False):
    """Classifica cada entidade como TP, FP ou FN numa manchete."""
    usadas, linhas = set(), []
    for p in previstas:
        par = next((j for j, o in enumerate(anotadas) if j not in usadas and o["rotulo"] == p["rotulo"] and (
            o["inicio"] < p["fim"] and p["inicio"] < o["fim"] if relaxada
            else o["texto"].lower() == normaliza(p["texto"]).lower())), None)
        if par is None:
            linhas.append(("FP", p["rotulo"], p["texto"]))
        else:
            usadas.add(par)
            linhas.append(("TP", p["rotulo"], p["texto"]))
    linhas += [("FN", o["rotulo"], o["texto"]) for j, o in enumerate(anotadas) if j not in usadas]
    return linhas


def avalia(n, relaxada=False):
    linhas = []
    for id_, anotadas in ouro.items():
        previstas = entidades(n, ouro_csv.set_index("id").loc[id_, "manchete"])
        linhas += [(id_, *l) for l in compara(previstas, anotadas, relaxada)]
    return pd.DataFrame(linhas, columns=["id", "tipo", "rotulo", "texto"])


def metricas(res):
    c = res.groupby(["rotulo", "tipo"]).size().unstack(fill_value=0).reindex(columns=["TP", "FP", "FN"], fill_value=0)
    c.loc["TOTAL (micro)"] = c.sum()
    c["precisão"] = c["TP"] / (c["TP"] + c["FP"]).where(lambda s: s > 0)
    c["revocação"] = c["TP"] / (c["TP"] + c["FN"]).where(lambda s: s > 0)
    c["F1"] = 2 * c["precisão"] * c["revocação"] / (c["precisão"] + c["revocação"])
    return c.reindex([r for r in ROTULOS if r in c.index] + ["TOTAL (micro)"]).round(2)


resultados = {nome: avalia(n) for nome, n in SISTEMAS.items()}
resumo = pd.DataFrame({
    nome: {
        "F1 exata": metricas(r).loc["TOTAL (micro)", "F1"],
        "F1 relaxada": metricas(avalia(SISTEMAS[nome], relaxada=True)).loc["TOTAL (micro)", "F1"],
        "F1 só ORG/PESSOA/LOCAL": metricas(r[r["rotulo"].isin(["ORG", "PESSOA", "LOCAL"])]).loc["TOTAL (micro)", "F1"],
    }
    for nome, r in resultados.items()
}).T
resumo

,F1 exata,F1 relaxada,F1 só ORG/PESSOA/LOCAL
A · sm puro,0.24,0.25,0.33
B · lg puro,0.37,0.38,0.50
C · B + carteira + regras,0.86,0.87,0.76
D · C + mercado,0.95,0.96,0.91


A última coluna compara os quatro só nos tipos que o spaCy conhece, para não punir A e B por não saberem o que é um ticker.

### Por tipo de entidade

In [6]:
pd.concat({nome: metricas(r)["F1"] for nome, r in resultados.items()}, axis=1)

,A · sm puro,B · lg puro,C · B + carteira + regras,D · C + mercado
rotulo,,,,
ORG,0.36,0.57,0.82,0.95
TICKER,NaN,NaN,1.00,1.00
PESSOA,0.38,0.58,0.67,0.88
LOCAL,0.24,0.27,0.47,0.67
VALOR,NaN,NaN,0.97,0.97
PERCENTUAL,NaN,NaN,1.00,1.00
PERIODO,NaN,NaN,1.00,1.00
TOTAL (micro),0.24,0.37,0.86,0.95


### Detalhe do C e do D

In [7]:
pd.concat({nome: metricas(resultados[nome]) for nome in ["C · B + carteira + regras", "D · C + mercado"]}, axis=1)

C · B + carteira + regras                                   \
tipo                                 TP  FP  FN precisão revocação    F1   
rotulo                                                                     
ORG                                  73   6  26     0.92      0.74  0.82   
TICKER                               37   0   0     1.00      1.00  1.00   
PESSOA                                7   6   1     0.54      0.88  0.67   
LOCAL                                 8  17   1     0.32      0.89  0.47   
VALOR                                17   0   1     1.00      0.94  0.97   
PERCENTUAL                           16   0   0     1.00      1.00  1.00   
PERIODO                              18   0   0     1.00      1.00  1.00   
TOTAL (micro)                       176  29  29     0.86      0.86  0.86   

              D · C + mercado                                   
tipo                       TP  FP  FN precisão revocação    F1  
rotulo                                                          
ORG                        91   2   8     0.98      0.92  0.95  
TICKER                     37   0   0     1.00      1.00  1.00  
PESSOA                      7   1   1     0.88      0.88  0.88  
LOCAL                       8   7   1     0.53      0.89  0.67  
VALOR                      17   0   1     1.00      0.94  0.97  
PERCENTUAL                 16   0   0     1.00      1.00  1.00  
PERIODO                    18   0   0     1.00      1.00  1.00  
TOTAL (micro)             194  10  11     0.95      0.95  0.95

## Ligação com a carteira (todas as 200 manchetes)

Para o radar, mais importante que acertar a fronteira de cada entidade é saber **de qual empresa da carteira** a manchete fala. A coluna `empresa` da base de teste diz qual é, então dá para medir nas 200 manchetes, e não só nas 60 anotadas.

**Limitação:** as 200 manchetes foram selecionadas no notebook 01 pela mesma lista de apelidos da carteira e, no notebook 02, só entraram manchetes com uma empresa. Por isso esta taxa mede só se o `EntityRuler` reproduz o filtro, não a cobertura em notícias novas. Uma empresa citada por um nome fora da lista (ex.: "dona da Vivo", "a petroleira") não seria encontrada.

In [8]:
manchetes["ents"] = [entidades(nlp, m) for m in manchetes["manchete"]]
manchetes["citadas"] = manchetes["ents"].apply(empresas_citadas)
manchetes["achou_empresa"] = [e in c for e, c in zip(manchetes["empresa"], manchetes["citadas"])]

print(f"Empresa certa encontrada em {manchetes['achou_empresa'].sum()} de {len(manchetes)} manchetes "
      f"({manchetes['achou_empresa'].mean():.1%})")
print(f"Manchetes que citam mais de uma empresa da carteira: {(manchetes['citadas'].str.len() > 1).sum()}")
manchetes.loc[~manchetes["achou_empresa"], ["id", "empresa", "manchete", "citadas"]]

Empresa certa encontrada em 200 de 200 manchetes (100.0%)
Manchetes que citam mais de uma empresa da carteira: 0


,id,empresa,manchete,citadas


## Erros do reconhecedor D

Entidades que faltaram (FN) e entidades a mais (FP), na avaliação exata.

In [9]:
erros = resultados["D · C + mercado"].query("tipo != 'TP'")
print(erros.groupby(["rotulo", "tipo"]).size().unstack(fill_value=0).to_string(), "\n")
erros.merge(ouro_csv[["id", "manchete"]], on="id").sort_values(["rotulo", "tipo"])[["rotulo", "tipo", "texto", "manchete"]]

tipo    FN  FP
rotulo        
LOCAL    1   7
ORG      8   2
PESSOA   1   1
VALOR    1   0 



,rotulo,tipo,texto,manchete
6,LOCAL,FN,Brasil,"Custo da Bolsa no Brasil pode ser 20 vezes maior do que em outros países, diz CEO da Base Exchange, futura concorren..."
3,LOCAL,FP,JP1,JP1: Raia Drogasil é condenada a pagar 10 milhões em multas a clientes
9,LOCAL,FP,Uzbequistão,Embraer entrega o primeiro C-390 Millennium à Força Aérea do Uzbequistão
11,LOCAL,FP,Fibrasil,"Telefônica Brasil (VIVT3), dona da Vivo, adquire Fibrasil por R$ 458,7 milhões"
13,LOCAL,FP,Rumo,"Depois do ‘rali Flavio’, JP Morgan eleva Rumo e corta Localiza; Motiva é ‘top pick’"
14,LOCAL,FP,Motiva,"Depois do ‘rali Flavio’, JP Morgan eleva Rumo e corta Localiza; Motiva é ‘top pick’"
18,LOCAL,FP,Prisma Capital,Itaú Unibanco fecha acordo para comprar 35% da Prisma Capital
20,LOCAL,FP,Prefeitura,"Prefeitura aplica multa de R$ 16,5 milhões à Sabesp"
1,ORG,FN,AllianzGI,Gestora alemã AllianzGI fecha acordo de distribuição com BTG Pactual Asset
2,ORG,FN,BTG Pactual Asset,Gestora alemã AllianzGI fecha acordo de distribuição com BTG Pactual Asset


### Exemplos anotados

In [10]:
for id_ in ["M0029", "M0089", "M0225", "M0294", "M0310", "M0480"]:
    displacy.render(nlp(normaliza(manchetes.set_index("id").loc[id_, "manchete"])), style="ent", jupyter=True,
                    options={"colors": CORES})

## Entidades + sentimento

Juntando o NER com os rótulos de sentimento revisados, já dá para responder perguntas que o chatbot vai receber, como "quais empresas tiveram mais notícias ruins?" e "quem aparece nas notícias ruins?".

In [11]:
radar = manchetes.assign(empresa=manchetes["citadas"]).explode("empresa").dropna(subset=["empresa"])
pd.crosstab(radar["empresa"], radar["rotulo"]).reindex(columns=["boa", "neutra", "ruim"], fill_value=0) \
    .sort_values(["ruim", "boa"], ascending=False).head(10)

rotulo,boa,neutra,ruim
empresa,,,
SANB11,3,4,4
ABEV3,4,5,3
PRIO3,3,5,3
BBAS3,2,5,3
RDOR3,6,3,2
RADL3,5,3,2
RENT3,5,4,2
B3SA3,8,0,1
PETR4,7,2,1


In [12]:
# Organizações fora da carteira (bancos, analistas, órgãos públicos) e pessoas, por sentimento da manchete
outros = [(l["rotulo"], e["rotulo"], e["texto"]) for _, l in manchetes.iterrows() for e in l["ents"]
          if e["rotulo"] in {"ORG", "PESSOA"} and not e["ticker"]]
outros = pd.DataFrame(outros, columns=["sentimento", "tipo", "entidade"])
outros.groupby(["entidade", "tipo"]).agg(manchetes=("sentimento", "size"),
                                          ruins=("sentimento", lambda s: (s == "ruim").sum())) \
    .sort_values(["manchetes", "ruins"], ascending=False).head(15)

,,manchetes,ruins
entidade,tipo,,
STF,ORG,4,0
União,ORG,4,0
JP Morgan,ORG,3,3
Bradsaúde,ORG,3,0
Safra,ORG,2,1
BlackRock,ORG,2,0
Citi,ORG,2,0
Emae,ORG,2,0
Latam,ORG,2,0


## Resultados salvos

In [13]:
saida = manchetes.assign(
    entidades=manchetes["ents"].apply(lambda es: "; ".join(f"{e['texto']}|{e['rotulo']}" for e in es)),
    empresas_citadas=manchetes["citadas"].str.join("|"),
)[["id", "empresa", "manchete", "rotulo", "entidades", "empresas_citadas", "achou_empresa"]]
saida.to_csv(SAIDA / "resultado_ner.csv", index=False)
pd.concat({nome: metricas(r) for nome, r in resultados.items()}).to_csv(SAIDA / "metricas_ner.csv")
print(f"Salvo em {SAIDA / 'resultado_ner.csv'} e {SAIDA / 'metricas_ner.csv'}")

Salvo em ..\Entregas\30-10-2026\resultado_ner.csv e ..\Entregas\30-10-2026\metricas_ner.csv


## Breve análise

**Resultado.** Nas 60 manchetes anotadas (205 entidades), o F1 por entidade (avaliação exata) foi:

| reconhecedor | F1 geral | F1 só ORG/PESSOA/LOCAL |
|---|---|---|
| A · spaCy `sm` puro | 0,24 | 0,33 |
| B · spaCy `lg` puro | 0,37 | 0,50 |
| C · B + carteira + regras | 0,86 | 0,76 |
| D · C + mercado | **0,95** | **0,91** |

**O que cada camada resolve.**
- **sm → lg:** o modelo maior acerta mais organizações (F1 de ORG 0,36 → 0,57). Mesmo assim, o spaCy puro confunde muito ORG com LOC: "Itaú" sai como `LOCAL` no `sm`, e o ticker "ITUB4" sai como `ORG` ou `LOCAL`. O WikiNER, base de treino do spaCy em português, tem pouco texto financeiro.
- **EntityRuler + regras (C):** tickers, valores, percentuais e períodos passam a ser reconhecidos com F1 entre 0,97 e 1,00, porque têm forma fixa (`[A-Z]{4}\d`, `R$ …`, `2T26`). O `id` do padrão liga cada nome à empresa: "Axia Energia", "ex-Eletrobras" e "ELET6" viram ELET3.
- **Contexto do mercado (D):** a regra "nome antes de (TICKER) é empresa" corrigiu "Cemig", "Casas Bahia", "Porto Seguro" e "Mercado Livre", que o modelo marcava como `LOCAL` ou não via. A lista de siglas tirou falsos positivos como "JCP" e "ROE". A precisão de ORG foi de 0,92 para 0,98.

**Erros que sobram.**
- Empresas fora da carteira, citadas sem ticker, ainda saem como `LOCAL` ("JP Morgan eleva **Rumo** e corta Localiza; **Motiva** é top pick", "**Prisma Capital**", "**Fibrasil**"). Por isso `LOCAL` tem a pior precisão (0,53).
- Fronteira: o NER marca "BTG Pactual" (o padrão da carteira) onde a anotação tem "BTG Pactual Asset", e "Força Aérea" + "Uzbequistão" onde anotamos uma organização só.
- Valor sem moeda ("pagar 10 milhões em multas") escapa da regra, que exige "R$" ou "US$".

**Ligação com a carteira.** O NER encontrou a empresa certa nas 200 manchetes, mas esse número é otimista por construção: as manchetes foram filtradas pela mesma lista de apelidos. Ele mostra que o `EntityRuler` reproduz o filtro, não que o NER cobre notícias novas.

**Integração.** Juntando entidades e sentimento, o radar já responde perguntas como "quem aparece nas notícias ruins?": o JP Morgan aparece em 3 manchetes, todas `ruim` (recomendações de venda e cortes), e Santander é a empresa com mais notícias ruins (4 de 11). É esse cruzamento que o chatbot vai consultar.

**Limitações.** A amostra anotada é pequena (60 manchetes) e as anotações foram sugeridas por IA e revisadas por nós. A lista de instituições do D foi montada com nomes que também aparecem na amostra, então o ganho do D é otimista. O próximo passo é medir em manchetes novas e testar um NER baseado em Transformer, como o BERTimbau ajustado para NER.